# **03 - Machine Learning Model EDA**
---

## <a id = 'summary'></a>**Summary**

---

## **Table of Content**

[Summary](#summary)

[Objective](#obj)

[Dataset Info](#data-info)

[Model Exploration Strategy](#strategy)

[Initial Setup](#setup)

[XGBoost](#xgboost)

---

## <a id ='obj'></a>**Objectives**
To evaluate and compare XGBoost, SARIMA, and CNN models under different configurations and feature selection strategies, while incorporating conformal prediction to quantify prediction uncertainty and assess the reliability of forecasting intervals.


## <a id ='data-info'></a> **Dataset Info**
* **Source:** UCI Machine Learning Repository
* **Title:** Appliances Energy Prediction
* **Time Range:** 11th January 2016 - 27th May 2016
* **Primary Target Variable:** `Appliances` (Watt-hour consumed)

## <a id = 'strategy'></a>**Model Exploration Strategy**
We're explore 3 models:
1. eXtreme Gradient Boosting (XGBoost)
2. Seasonal Autoregressive Integrated Moving Average (SARIMA)
3. Convolutional Neural Network (CNN)

And as these models are all of different model family (*gradient boosting, statistical and deep learning family, respectively)*, they have to be treated very differently. As such, we'll explore these models sequentially.

For every model exploration, we'll keep it within these stages:
1. Feature preprocessing
2. Hyperparameter Tuning

, which will be tailored to fit the currently explored model. Description for each model stages can be found on top of sections named after the model.

## **Initial Setup**

### Import Necessary Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys
from sklearn.preprocessing import LabelEncoder
from mrmr import mrmr_regression
import numpy as np

# Add the root directory (one level up from notebooks/) to Python's path
root_dir = Path.cwd().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from src.preprocess import preprocess_energy, feature_engineering


### Import Raw Dataset

In [ ]:
PROJECT_ROOT = Path.cwd().parent

# Import csv file
energy = pd.read_csv(f'{PROJECT_ROOT}/data/raw/energydata_complete.csv')

### Data Preprocessing and Formation of Feature Space

In [ ]:
# Data preprocessing
energy = preprocess_energy(energy)

# Form feature through feature engineering
energy = feature_engineering(energy)

### Inspection

In [ ]:
energy.info()

## **Feature Reduction**

Let's see if we can reduce more information. We'll use a model-agnostic technique, specifically the **Minimum Redundancy Maximum Relevance (mRMR)**.

**Justification**:

* **Model-agnostic**: As we'll use the same reduced features for 3 different model, this is non-negotiable.

* **Does not require scaling**: Tree-based model (XGBoost) prefers raw values over scaled. Other models that prefers scaling will just be preprocessed separately in their own isolated DataFrame.

* **Suits time-series data**: mRMR accounts for autocorrelation and time-aware distance measure, which accomodates time-series forecasting tasks.

For now, we'll choose 15 features as a starting point. We'll test whether this number is optimal in later stage, after we've established the best model.

In [ ]:
# Extract features and target
X = energy.drop(columns = ['Appliances'])
y = energy['Appliances']

In [ ]:
# Encode month
le_month = LabelEncoder()
X['month'] = le_month.fit_transform(X['month'])

In [ ]:
# Convert returned numpy array into DataFrame (because mrmr_regression requires it)
X = pd.DataFrame(X)

In [ ]:
# Run mRMR feature selection
selected_features = mrmr_regression(X = X, y=y, K=15)

In [ ]:
# Separate the selected features
reduced_X = X[selected_features]

## <a id = 'xgboost'></a>**XGBoost**

### Stages Decsription
**Feature preprocessing**

These actions will be done to accomodate XGBoost model:
* Cyclical time encoding
    
    **Justification**: Represents the circular nature of time, so 23:00 and 00:00 are treated as nearby times.

* Time-based train-test split

    **Justification**: As this is a time-series dataset, where row order means something, we will split by chronological order rather than shuffling.

**Hyperparameter Tuning**

These hyperparameters will be searched using **Bayesian Optimization**:
* 'max_depth': [3, 5, 7, 9]
* 'min_child_weight': [1, 3, 5, 7]
* 'subsample': [0.6, 0.7, 0.8, 0.9, 1.0]
* 'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0]
* 'learning_rate': [0.01, 0.05, 0.1, 0.2]

### Feature Preprocessing

*Cyclical Time Encoding*

In [ ]:
# Specify target column to apply cyclical encoding
col = 'hour'

# Apply both sine and cosine cyclical encoding
reduced_X[f"{col}_sin"] = np.sin(2 * np.pi * reduced_X[col] / 24 * 2)
reduced_X[f"{col}_cos"] = np.cos(2 * np.pi * reduced_X[col] / 24 * 2)

# Drop the original raw column
reduced_X = reduced_X.drop(columns=[col])

*Time-based Train-Test Split (80/20)*

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# 80-20 train-test split, no shuffling (to preserve  chronlogical order)
X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(
    reduced_X, y, test_size=0.2, random_state=42, shuffle = False
)

### Hyperparameter Tuning

In [ ]:
from xgboost import XGBRegressor
import optuna
from sklearn.model_selection import cross_val_score


In [ ]:
from sklearn.metrics import get_scorer_names

scorer = get_scorer_names()
for item in scorer:
    print(item)

In [ ]:
def objective(trial):

    # Disable Optuna's verbose logging to keep notebook output clean
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    # Define hyperparameter grid
    params = {
        'max_depth' : trial.suggest_categorical('max_depth', [3, 5, 7, 9]),
        'min_child_weight': trial.suggest_categorical ('min_child_weight', [1, 3, 5, 7]) ,
        'subsample':trial.suggest_categorical('subsample', [0.6, 0.7, 0.8, 0.9, 1.0]) ,
        'colsample_bytree': trial.suggest_categorical('colsample_bytree', [0.6, 0.7, 0.8, 0.9, 1.0]) ,
        'learning_rate': trial.suggest_categorical('learning_rate', [0.01, 0.05, 0.1, 0.2]) ,
        'random_state': 42
    }

    
    # Initialize the model with suggested parameters
    xgb = XGBRegressor(**params)
    
    # Evaluate using Cross-Validation
    score = cross_val_score(
        xgb,
        X_train_xgb,
        y_train_xgb, 
        cv=3,
        scoring = 'neg_root_mean_squared_error'
    ).mean()
    
    # Return the metric to maximize
    return score



In [ ]:
# Initiate search
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

In [ ]:
# Print best results
print("Best RMSE:", study.best_value)

In [ ]:
import joblib

In [ ]:
best_params = study.best_params

xgb_best = XGBRegressor(
    **best_params,
    random_state=42
)

xgb_best.fit(X_train_xgb, y_train_xgb)

In [ ]:
joblib.dump(xgb_best, f"{PROJECT_ROOT}/models/xgb_best.pkl")

## <a id = 'sarima'></a>**SARIMA**

### Stages Decsription
**Feature preprocessing**

These actions will be done to accomodate SARIMA:

* Augmented Dickey-Fuller (ADF) test

    **Justification**: SARIMA assumes series stationarity, so this test checks it formally. If our series is not stationary (or in another words, the mean and variance changes over time), we'll have to perform differencing.

**Hyperparameter Tuning**

The hyperparameter tuning for this model is more straightforward, as we only need to call 'auto_arima' method from 'pmdarima' package to automate the process.

## <a id = 'cnn'></a>**CNN**

### Stages Decsription
**Feature preprocessing**

**Hyperparameter Tuning**
